# Steering sweep for the dose-response plot (GPU)

Generates replies to 20 everyday prompts at strengths 0, 0.25, 0.5, 1.0 and 1.5 with the **desperate** vector at block 23. Each prompt keeps one seed across all strengths, so every steered reply has a matched baseline (strength 0).

It also writes a second baseline per prompt with a different seed. The judge scores it against the first baseline as a control: it should come out near 50.

Output: `results/steering/sweep_desperate.json`, scored afterwards by `judge.ipynb` on a laptop.

**Run on:** an NVIDIA GPU, from inside `notebooks/`, after `steering.ipynb` passes its checks.

In [1]:
import json, sys, time
from pathlib import Path
import torch

sys.path.insert(0, str(Path.cwd().parent))
from src.steering import LAYER, MODEL_REVISION, load_model_8b, load_vector, residual_norms, steer, generate

RUN_DIR = Path("../emotion_vector_pipeline/outputs/emotion_extraction/ab1333f49024705c")
STORIES = Path("../emotion_vector_pipeline/data/emotion_stories_qwen_full1120.jsonl")
OUT = Path("../results/steering")
OUT.mkdir(parents=True, exist_ok=True)

EMOTION = "desperate"
STRENGTHS = [0, 0.25, 0.5, 1.0, 1.5]
PROMPTS = [
    "Write a short paragraph about your morning.",
    "Describe your weekend plans in a few sentences.",
    "Write a short note to a friend about your week.",
    "Describe a trip to the grocery store.",
    "Write a few sentences about your job.",
    "Describe the room you are sitting in.",
    "Write a short paragraph about cooking dinner.",
    "Describe your commute to work.",
    "Write a short paragraph about a rainy afternoon.",
    "Tell me about a recent phone call with your family.",
    "Describe a walk in the park.",
    "Write a short paragraph about waiting for a bus.",
    "Describe your plans for tomorrow.",
    "Write a few sentences about cleaning your apartment.",
    "Write a short paragraph about checking your email.",
    "Write a few sentences about your favorite meal.",
    "Describe a day at the library.",
    "Write a short paragraph about doing laundry.",
    "Describe an evening at home.",
    "Write a few sentences about a new hobby.",
]
print(len(PROMPTS), "prompts x", len(STRENGTHS), "strengths")

20 prompts x 5 strengths


In [2]:
tok, model = load_model_8b()
v = load_vector(RUN_DIR / "emotion_vectors.npz", EMOTION)

#same residual-norm measurement as steering.ipynb
rows = [json.loads(line) for line in STORIES.open()]
texts = []
for e in sorted({r["emotion"] for r in rows}):
    texts += [r["text"] for r in rows if r["emotion"] == e and r["split"] == "validation"][:2]
_, rest = residual_norms(model, tok, texts)
NORM = rest.mean().item()
print("residual norm at block", LAYER, "=", round(NORM, 1))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

residual norm at block 23 = 198.0


In [3]:
records = []
start = time.time()
for i, prompt in enumerate(PROMPTS):
    seed = 1000 + i
    rec = dict(prompt=prompt, seed=seed, outputs={})
    for c in STRENGTHS:
        with steer(model, v, c, NORM):
            rec["outputs"][str(c)] = generate(model, tok, prompt, seed=seed, max_new_tokens=100)
    #control: a second baseline with a different seed
    rec["baseline_other_seed"] = generate(model, tok, prompt, seed=seed + 500, max_new_tokens=100)
    records.append(rec)
    print(f"{i + 1}/{len(PROMPTS)} done, {time.time() - start:.0f}s")

#strength 0 is a zero vector, so the first prompt's strength-0 reply must match a run with no hook
assert records[0]["outputs"]["0"] == generate(model, tok, PROMPTS[0], seed=1000, max_new_tokens=100)

1/20 done, 31s


2/20 done, 59s


3/20 done, 93s


4/20 done, 127s


5/20 done, 153s


6/20 done, 184s


... (10 outputs left out)

17/20 done, 517s


18/20 done, 548s


19/20 done, 581s


20/20 done, 609s


In [4]:
sweep = dict(emotion=EMOTION, layer=LAYER, residual_norm=NORM, model_revision=MODEL_REVISION,
             strengths=STRENGTHS, max_new_tokens=100, temperature=0.7, top_p=0.9, records=records)
(OUT / f"sweep_{EMOTION}.json").write_text(json.dumps(sweep, indent=2))
print("saved", OUT / f"sweep_{EMOTION}.json")

#a quick look at one prompt across strengths
for c in STRENGTHS:
    print(f"--- {c} ---\n{records[0]['outputs'][str(c)]}\n")

saved ../results/steering/sweep_desperate.json
--- 0 ---
My morning starts with the gentle hum of my coffee maker, signaling the beginning of the day. I stretch lazily in bed, savoring the warmth of my cup as I step into the quiet stillness of the early hours. The sun is just beginning to rise, casting a soft golden glow through the curtains. I take a moment to breathe, reflect, and prepare for the day ahead with a sense of calm and purpose.

--- 0.25 ---
Every morning, I wake up at 6:30 AM. I start by checking my phone to see the weather and news updates. Then I make my bed and do a quick workout to get my day started. After that, I make breakfast, usually something quick like oatmeal or toast. I try to read a few news articles or listen to a podcast while I eat. By 8:00 AM, I'm ready to start work or school. I try to stay positive and

--- 0.5 ---
I woke up at 6:30 AM, still half-asleep, and tried to get out of bed. I made coffee, then got dressed and tried to remember what day it is